# 03 — Feature Engineering
## Nexora Supply Chain Analytics Platform

**Business objective:** enrich the cleaned DataCo dataset with consistent analytical features for supply-chain performance, customer segmentation, profitability analysis, MySQL warehousing, and Tableau dashboards.

**Input**
- `dataco_supply_chain_cleaned.csv`

**Main output**
- `dataco_supply_chain_analytics_ready.csv`
- `feature_dictionary.csv`
- `feature_engineering_validation.csv`

**Grain remains unchanged:** one row represents one order item.

## 1. Feature-engineering principles

1. Preserve the original order-item grain.
2. Avoid arbitrary imputation of sales, profit, quantity, or dates.
3. Create transparent and explainable business features.
4. Protect against division by zero and invalid dates.
5. Validate row count, primary key, value ranges, and feature consistency.

In [ ]:
import importlib.util
import subprocess
import sys

required = ["pandas", "numpy", "matplotlib", "plotly", "openpyxl"]
for package in required:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Load the cleaned dataset

The notebook searches common Colab paths and also supports manual upload.

In [ ]:
FILE_CANDIDATES = [
    "/content/nexora_outputs/dataco_supply_chain_cleaned.csv",
    "/content/dataco_supply_chain_cleaned.csv",
    "/mnt/data/dataco_supply_chain_cleaned.csv",
]

def resolve_input_file(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    try:
        from google.colab import files
        print("Upload dataco_supply_chain_cleaned.csv")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")
        return next(iter(uploaded.keys()))
    except ImportError as exc:
        raise FileNotFoundError(
            "Cleaned CSV not found. Update FILE_CANDIDATES with the correct path."
        ) from exc

def read_csv_robust(path):
    for encoding in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError("Unable to detect CSV encoding.")

FILE_PATH = resolve_input_file(FILE_CANDIDATES)
df = read_csv_robust(FILE_PATH)

input_rows, input_cols = df.shape
print(f"Input file : {FILE_PATH}")
print(f"Input shape: {df.shape}")
display(df.head())

Upload dataco_supply_chain_cleaned.csv


Saving dataco_supply_chain_cleaned.csv to dataco_supply_chain_cleaned.csv
Input file : dataco_supply_chain_cleaned.csv
Input shape: (180519, 59)


,payment_type,days_for_shipping,days_for_shipment,order_profit,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_customer_id,order_date,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_state,order_status,order_zipcode,product_card_id,product_category_id,product_name,product_price,product_status,shipping_date,shipping_mode,is_late_delivery,profit_margin_pct,gross_sales_before_discount,profitability_status,order_year,order_quarter,order_month,order_month_name,order_week,order_day,order_day_name,order_date_key,shipping_date_key
0,CASH,2,4,88.7900,239.9800,Advance Shipping,0,43,Camping & Hiking,Hickory,Ee. Uu.,11599,Consumer,Nc,"28,601.0000",7,Fan Shop,35.7767,-81.3626,LATAM,Mexico City,México,11599,2015-01-01 00:00:00,1,957,60.0000,0.2000,1,299.9800,0.3700,1,299.9800,239.9800,88.7900,Central America,Distrito Federal,CLOSED,NaN,957,43,Diamondback Women's Serene Classic Comfort Bi,299.9800,0,2015-01-03 00:00:00,Standard Class,0,29.5986,359.9800,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150103
1,PAYMENT,3,4,91.1800,193.9900,Advance Shipping,0,48,Water Sports,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",7,Fan Shop,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,1073,6.0000,0.0300,2,199.9900,0.4700,1,199.9900,193.9900,91.1800,South America,Risaralda,PENDING_PAYMENT,NaN,1073,48,Pelican Sunstream 100 Kayak,199.9900,0,2015-01-04 00:21:00,Standard Class,0,45.5923,205.9900,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
2,PAYMENT,3,4,68.2500,227.5000,Advance Shipping,0,24,Women's Apparel,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",5,Golf,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,502,22.5000,0.0900,3,50.0000,0.3000,5,250.0000,227.5000,68.2500,South America,Risaralda,PENDING_PAYMENT,NaN,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.0000,0,2015-01-04 00:21:00,Standard Class,0,27.3000,272.5000,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
3,PAYMENT,3,4,36.4700,107.8900,Advance Shipping,0,18,Men's Footwear,Chicago,Ee. Uu.,256,Consumer,Il,"60,625.0000",4,Apparel,41.8327,-87.9805,LATAM,Dos Quebradas,Colombia,256,2015-01-01 00:21:00,2,403,22.1000,0.1700,4,129.9900,0.3400,1,129.9900,107.8900,36.4700,South America,Risaralda,PENDING_PAYMENT,NaN,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.9900,0,2015-01-04 00:21:00,Standard Class,0,28.0560,152.0900,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150104
4,CASH,5,4,4.1000,40.9800,Late Delivery,1,40,Accessories,San Antonio,Ee. Uu.,8827,Home Office,Tx,"78,240.0000",6,Outdoors,29.5200,-98.6374,LATAM,Dos Quebradas,Colombia,8827,2015-01-01 01:03:00,4,897,9.0000,0.1800,5,24.9900,0.1000,2,49.9800,40.9800,4.1000,South America,Risaralda,CLOSED,NaN,897,40,Team Golf New England Patriots Putter Grip,24.9900,0,2015-01-06 01:03:00,Standard Class,1,8.2033,58.9800,Profitable,2015,Q1,1,January,1,1,Thursday,20150101,20150106


## 3. Standardize required data types

In [ ]:
date_columns = [c for c in ["order_date", "shipping_date"] if c in df.columns]
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_candidates = [
    "days_for_shipping_real", "days_for_shipment_scheduled",
    "order_profit", "order_profit_per_order", "sales",
    "sales_per_customer", "order_item_discount",
    "order_item_discount_rate", "order_item_product_price",
    "order_item_profit_ratio", "order_item_quantity",
    "order_item_total", "product_price", "latitude", "longitude",
    "shipping_delay_days", "profit_margin_pct",
    "gross_sales_before_discount", "late_delivery_risk"
]
for col in [c for c in numeric_candidates if c in df.columns]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dates parsed :", date_columns)
print("Numeric fields standardized.")

Dates parsed : ['order_date', 'shipping_date']
Numeric fields standardized.


## 4. Calendar features

Calendar attributes support trend analysis, seasonality, weekday analysis, and date dimensions.

In [ ]:
if "order_date" in df.columns:
    iso = df["order_date"].dt.isocalendar()
    df["order_year"] = df["order_date"].dt.year.astype("Int64")
    df["order_quarter_number"] = df["order_date"].dt.quarter.astype("Int64")
    df["order_quarter"] = "Q" + df["order_quarter_number"].astype("string")
    df["order_month"] = df["order_date"].dt.month.astype("Int64")
    df["order_month_name"] = df["order_date"].dt.month_name()
    df["order_year_month"] = df["order_date"].dt.to_period("M").astype("string")
    df["order_week"] = iso.week.astype("Int64")
    df["order_day"] = df["order_date"].dt.day.astype("Int64")
    df["order_day_name"] = df["order_date"].dt.day_name()
    df["is_weekend_order"] = df["order_date"].dt.dayofweek.isin([5, 6]).astype("Int8")
    df["order_date_key"] = df["order_date"].dt.strftime("%Y%m%d").astype("Int64")

if "shipping_date" in df.columns:
    df["shipping_year_month"] = df["shipping_date"].dt.to_period("M").astype("string")
    df["shipping_date_key"] = df["shipping_date"].dt.strftime("%Y%m%d").astype("Int64")

print("Calendar features created.")

Calendar features created.


## 5. Delivery and service-level features

These features measure lateness, schedule variance, reliability, and service performance.

In [ ]:
required_shipping = {"days_for_shipping_real", "days_for_shipment_scheduled"}
if required_shipping.issubset(df.columns):
    df["shipping_delay_days"] = (
        df["days_for_shipping_real"] - df["days_for_shipment_scheduled"]
    )

    df["absolute_schedule_variance_days"] = df["shipping_delay_days"].abs()

    df["delivery_performance"] = np.select(
        [
            df["shipping_delay_days"] > 0,
            df["shipping_delay_days"] == 0,
            df["shipping_delay_days"] < 0,
        ],
        ["Late", "On Schedule", "Early"],
        default="Unknown",
    )

    df["is_on_time_delivery"] = df["shipping_delay_days"].le(0).astype("Int8")
    df["is_late_delivery"] = df["shipping_delay_days"].gt(0).astype("Int8")

    df["shipping_efficiency_ratio"] = np.where(
        df["days_for_shipment_scheduled"].gt(0),
        df["days_for_shipping_real"] / df["days_for_shipment_scheduled"],
        np.nan,
    )

    df["delivery_delay_severity"] = pd.cut(
        df["shipping_delay_days"],
        bins=[-np.inf, -1, 0, 1, 3, np.inf],
        labels=["Early", "On Schedule", "1 Day Late", "2–3 Days Late", "4+ Days Late"],
    )

    df["shipping_speed_segment"] = pd.cut(
        df["days_for_shipping_real"],
        bins=[-np.inf, 2, 4, 6, np.inf],
        labels=["Express", "Standard", "Slow", "Very Slow"],
    )

print("Delivery features created.")

Delivery features created.


## 6. Financial and commercial features

Financial features help identify profitable sales, loss-making items, discount dependence, and order-value tiers.

In [ ]:
profit_col = (
    "order_profit_per_order"
    if "order_profit_per_order" in df.columns
    else "order_profit" if "order_profit" in df.columns else None
)

if {"sales", "order_item_discount"}.issubset(df.columns):
    df["gross_sales_before_discount"] = df["sales"] + df["order_item_discount"]
    df["discount_pct_calculated"] = np.where(
        df["gross_sales_before_discount"].gt(0),
        df["order_item_discount"] / df["gross_sales_before_discount"] * 100,
        np.nan,
    )

if "order_item_discount_rate" in df.columns:
    rate = df["order_item_discount_rate"].copy()
    df["discount_rate_pct"] = np.where(rate.abs().le(1), rate * 100, rate)

if profit_col and "sales" in df.columns:
    df["profit_margin_pct"] = np.where(
        df["sales"].ne(0), df[profit_col] / df["sales"] * 100, np.nan
    )
    df["profitability_status"] = np.select(
        [df[profit_col] > 0, df[profit_col] == 0, df[profit_col] < 0],
        ["Profitable", "Break Even", "Loss"],
        default="Unknown",
    )
    df["is_profitable_item"] = df[profit_col].gt(0).astype("Int8")
    df["is_loss_item"] = df[profit_col].lt(0).astype("Int8")

if "sales" in df.columns:
    positive_sales = df.loc[df["sales"].notna(), "sales"]
    if positive_sales.nunique() >= 4:
        df["sales_value_tier"] = pd.qcut(
            positive_sales.rank(method="first"),
            q=4,
            labels=["Low", "Medium", "High", "Premium"],
        ).reindex(df.index)

if "profit_margin_pct" in df.columns:
    df["margin_band"] = pd.cut(
        df["profit_margin_pct"],
        bins=[-np.inf, 0, 10, 20, 30, np.inf],
        labels=["Negative", "0–10%", "10–20%", "20–30%", "30%+"],
    )

discount_source = (
    "discount_rate_pct"
    if "discount_rate_pct" in df.columns
    else "discount_pct_calculated" if "discount_pct_calculated" in df.columns else None
)
if discount_source:
    df["discount_band"] = pd.cut(
        df[discount_source],
        bins=[-np.inf, 0, 5, 10, 20, np.inf],
        labels=["No Discount", "0–5%", "5–10%", "10–20%", "20%+"],
    )

print("Financial features created.")

Financial features created.


## 7. Order- and customer-behaviour features

Aggregate attributes are mapped back to each order item without changing the dataset grain.

In [ ]:
if "order_id" in df.columns:
    order_agg_dict = {}
    if "sales" in df.columns:
        order_agg_dict["sales"] = "sum"
    if profit_col:
        order_agg_dict[profit_col] = "sum"
    if "order_item_quantity" in df.columns:
        order_agg_dict["order_item_quantity"] = "sum"
    if "order_item_id" in df.columns:
        order_agg_dict["order_item_id"] = "count"

    order_metrics = df.groupby("order_id", dropna=False).agg(order_agg_dict)
    rename_order = {}
    if "sales" in order_metrics.columns:
        rename_order["sales"] = "order_total_sales"
    if profit_col in order_metrics.columns:
        rename_order[profit_col] = "order_total_profit"
    if "order_item_quantity" in order_metrics.columns:
        rename_order["order_item_quantity"] = "order_total_quantity"
    if "order_item_id" in order_metrics.columns:
        rename_order["order_item_id"] = "order_item_count"
    order_metrics = order_metrics.rename(columns=rename_order)
    df = df.join(order_metrics, on="order_id")

    if {"order_total_profit", "order_total_sales"}.issubset(df.columns):
        df["order_profit_margin_pct"] = np.where(
            df["order_total_sales"].ne(0),
            df["order_total_profit"] / df["order_total_sales"] * 100,
            np.nan,
        )

customer_key = (
    "customer_id" if "customer_id" in df.columns
    else "order_customer_id" if "order_customer_id" in df.columns else None
)

if customer_key:
    customer_group = df.groupby(customer_key, dropna=False)

    if "order_id" in df.columns:
        df["customer_order_count"] = customer_group["order_id"].transform("nunique")
    if "sales" in df.columns:
        df["customer_lifetime_sales"] = customer_group["sales"].transform("sum")
        df["customer_average_item_sales"] = customer_group["sales"].transform("mean")
    if profit_col:
        df["customer_lifetime_profit"] = customer_group[profit_col].transform("sum")
    if "order_date" in df.columns:
        df["customer_first_order_date"] = customer_group["order_date"].transform("min")
        df["customer_last_order_date"] = customer_group["order_date"].transform("max")
        df["customer_tenure_days"] = (
            df["customer_last_order_date"] - df["customer_first_order_date"]
        ).dt.days.astype("Int64")

    if "customer_order_count" in df.columns:
        df["customer_frequency_segment"] = pd.cut(
            df["customer_order_count"],
            bins=[0, 1, 2, 5, 10, np.inf],
            labels=["One-Time", "Occasional", "Repeat", "Loyal", "VIP"],
        )

print("Order and customer features created.")

Order and customer features created.


## 8. Risk and management flags

Composite flags make dashboard filtering and exception analysis easier.

In [ ]:
conditions = []

if "is_late_delivery" in df.columns:
    conditions.append(df["is_late_delivery"].eq(1))
if "is_loss_item" in df.columns:
    conditions.append(df["is_loss_item"].eq(1))

if conditions:
    combined_risk = conditions[0].copy()
    for condition in conditions[1:]:
        combined_risk = combined_risk | condition
    df["requires_management_attention"] = combined_risk.astype("Int8")

if {"is_late_delivery", "is_loss_item"}.issubset(df.columns):
    df["operational_risk_segment"] = np.select(
        [
            df["is_late_delivery"].eq(1) & df["is_loss_item"].eq(1),
            df["is_late_delivery"].eq(1) & df["is_loss_item"].eq(0),
            df["is_late_delivery"].eq(0) & df["is_loss_item"].eq(1),
        ],
        ["Late and Loss", "Late Only", "Loss Only"],
        default="Healthy",
    )

print("Risk features created.")

Risk features created.


## 9. Feature dictionary

In [ ]:
feature_descriptions = {
    "order_year_month": "Monthly period derived from order date.",
    "is_weekend_order": "1 when order date falls on Saturday or Sunday.",
    "shipping_delay_days": "Actual shipping days minus scheduled shipping days.",
    "absolute_schedule_variance_days": "Absolute difference between actual and scheduled shipping days.",
    "is_on_time_delivery": "1 when shipment is early or on schedule.",
    "is_late_delivery": "1 when actual shipping duration exceeds schedule.",
    "shipping_efficiency_ratio": "Actual shipping days divided by scheduled shipping days.",
    "delivery_delay_severity": "Categorical severity of schedule variance.",
    "shipping_speed_segment": "Shipping-speed category based on actual duration.",
    "gross_sales_before_discount": "Sales plus item discount amount.",
    "discount_pct_calculated": "Discount amount as a percentage of gross sales.",
    "discount_rate_pct": "Standardized discount rate expressed as percentage.",
    "profit_margin_pct": "Item profit divided by sales.",
    "profitability_status": "Profitable, break-even, or loss classification.",
    "sales_value_tier": "Quartile-based item sales tier.",
    "margin_band": "Profit-margin category.",
    "discount_band": "Discount-intensity category.",
    "order_total_sales": "Total sales across all items in an order.",
    "order_total_profit": "Total profit across all items in an order.",
    "order_total_quantity": "Total quantity across all items in an order.",
    "order_item_count": "Number of order-item rows within an order.",
    "order_profit_margin_pct": "Total order profit divided by total order sales.",
    "customer_order_count": "Distinct orders associated with a customer.",
    "customer_lifetime_sales": "Total item sales associated with a customer.",
    "customer_lifetime_profit": "Total item profit associated with a customer.",
    "customer_tenure_days": "Days between customer's first and last order.",
    "customer_frequency_segment": "Customer segment based on order frequency.",
    "requires_management_attention": "1 when an item is late, loss-making, or both.",
    "operational_risk_segment": "Combined lateness and profitability risk category.",
}

engineered_features = [c for c in feature_descriptions if c in df.columns]
feature_dictionary = pd.DataFrame({
    "feature_name": engineered_features,
    "data_type": [str(df[c].dtype) for c in engineered_features],
    "description": [feature_descriptions[c] for c in engineered_features],
    "missing_count": [int(df[c].isna().sum()) for c in engineered_features],
    "missing_pct": [round(df[c].isna().mean() * 100, 4) for c in engineered_features],
})

display(feature_dictionary)

,feature_name,data_type,description,missing_count,missing_pct
0,order_year_month,string,Monthly period derived from order date.,0,0.0000
1,is_weekend_order,Int8,1 when order date falls on Saturday or Sunday.,0,0.0000
2,is_late_delivery,int64,1 when actual shipping duration exceeds schedule.,0,0.0000
3,gross_sales_before_discount,float64,Sales plus item discount amount.,0,0.0000
4,discount_pct_calculated,float64,Discount amount as a percentage of gross sales.,0,0.0000
5,discount_rate_pct,float64,Standardized discount rate expressed as percen...,0,0.0000
6,profit_margin_pct,float64,Item profit divided by sales.,0,0.0000
7,profitability_status,object,"Profitable, break-even, or loss classification.",0,0.0000
8,sales_value_tier,category,Quartile-based item sales tier.,0,0.0000
9,margin_band,category,Profit-margin category.,0,0.0000


## 10. Validation

The final dataset must preserve row count and order-item uniqueness while passing logical consistency checks.

In [ ]:
validation_rows = []

def add_check(check_name, passed, observed, expected):
    validation_rows.append({
        "check": check_name,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": observed,
        "expected": expected,
    })

add_check(
    "Row count preserved",
    len(df) == input_rows,
    len(df),
    input_rows,
)

if "order_item_id" in df.columns:
    add_check(
        "Order item ID has no missing values",
        df["order_item_id"].isna().sum() == 0,
        int(df["order_item_id"].isna().sum()),
        0,
    )
    add_check(
        "Order item ID is unique",
        df["order_item_id"].duplicated().sum() == 0,
        int(df["order_item_id"].duplicated().sum()),
        0,
    )

if {"shipping_delay_days", "days_for_shipping_real", "days_for_shipment_scheduled"}.issubset(df.columns):
    mismatch = (
        df["shipping_delay_days"]
        != df["days_for_shipping_real"] - df["days_for_shipment_scheduled"]
    ).fillna(False).sum()
    add_check("Shipping delay formula is consistent", mismatch == 0, int(mismatch), 0)

if "is_late_delivery" in df.columns:
    invalid_binary = (~df["is_late_delivery"].isin([0, 1])).sum()
    add_check("Late-delivery flag is binary", invalid_binary == 0, int(invalid_binary), 0)

if "profit_margin_pct" in df.columns:
    infinite_margin = np.isinf(df["profit_margin_pct"]).sum()
    add_check("Profit margin contains no infinity", infinite_margin == 0, int(infinite_margin), 0)

validation_report = pd.DataFrame(validation_rows)
display(validation_report)

failed_checks = validation_report.loc[validation_report["status"] == "FAIL"]
if len(failed_checks):
    print("Review failed checks before loading the dataset into MySQL.")
else:
    print("All feature-engineering checks passed.")

,check,status,observed,expected
0,Row count preserved,PASS,180519,180519
1,Order item ID has no missing values,PASS,0,0
2,Order item ID is unique,PASS,0,0
3,Late-delivery flag is binary,PASS,0,0
4,Profit margin contains no infinity,PASS,0,0


All feature-engineering checks passed.


## 11. Feature preview and summary

In [ ]:
preview_cols = [
    c for c in [
        "order_item_id", "order_id", "order_date", "sales",
        "profit_margin_pct", "discount_band", "shipping_delay_days",
        "delivery_delay_severity", "is_late_delivery",
        "order_total_sales", "customer_order_count",
        "customer_frequency_segment", "operational_risk_segment"
    ] if c in df.columns
]
display(df[preview_cols].head(10))

summary = pd.DataFrame({
    "metric": [
        "input_rows", "output_rows", "input_columns", "output_columns",
        "engineered_features_documented", "validation_checks",
        "validation_checks_passed"
    ],
    "value": [
        input_rows, len(df), input_cols, df.shape[1],
        len(engineered_features), len(validation_report),
        int(validation_report["status"].eq("PASS").sum())
    ],
})
display(summary)

,order_item_id,order_id,order_date,sales,profit_margin_pct,discount_band,is_late_delivery,order_total_sales,customer_order_count,customer_frequency_segment,operational_risk_segment
0,1,1,2015-01-01 00:00:00,299.9800,29.5986,10–20%,0,299.9800,3,Repeat,Healthy
1,2,2,2015-01-01 00:21:00,199.9900,45.5923,0–5%,0,579.9800,10,Loyal,Healthy
2,3,2,2015-01-01 00:21:00,250.0000,27.3000,5–10%,0,579.9800,10,Loyal,Healthy
3,4,2,2015-01-01 00:21:00,129.9900,28.0560,10–20%,0,579.9800,10,Loyal,Healthy
4,5,4,2015-01-01 01:03:00,49.9800,8.2033,10–20%,1,699.8500,4,Repeat,Late Only
5,6,4,2015-01-01 01:03:00,299.9500,8.7115,0–5%,1,699.8500,4,Repeat,Late Only
6,7,4,2015-01-01 01:03:00,150.0000,40.1800,10–20%,1,699.8500,4,Repeat,Late Only
7,8,4,2015-01-01 01:03:00,199.9200,16.8017,10–20%,1,699.8500,4,Repeat,Late Only
8,9,5,2015-01-01 01:24:00,299.9800,30.7487,10–20%,1,"1,129.8600",4,Repeat,Late Only
9,10,5,2015-01-01 01:24:00,299.9500,48.0013,No Discount,1,"1,129.8600",4,Repeat,Late Only


,metric,value
0,input_rows,180519
1,output_rows,180519
2,input_columns,59
3,output_columns,85
4,engineered_features_documented,23
5,validation_checks,5
6,validation_checks_passed,5


## 12. Export analytics-ready dataset

Date values are exported in a MySQL-compatible timestamp format. Categorical columns are converted to text before CSV export.

In [ ]:
OUTPUT_DIR = Path("/content/nexora_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "dataco_supply_chain_analytics_ready.csv"
FEATURE_DICTIONARY_CSV = OUTPUT_DIR / "feature_dictionary.csv"
VALIDATION_CSV = OUTPUT_DIR / "feature_engineering_validation.csv"

export_df = df.copy()

datetime_export_cols = [
    c for c in [
        "order_date", "shipping_date",
        "customer_first_order_date", "customer_last_order_date"
    ] if c in export_df.columns
]
for col in datetime_export_cols:
    export_df[col] = export_df[col].dt.strftime("%Y-%m-%d %H:%M:%S")

category_cols = export_df.select_dtypes(include=["category"]).columns
for col in category_cols:
    export_df[col] = export_df[col].astype("string")

numeric_round_cols = export_df.select_dtypes(include=["float32", "float64"]).columns
export_df[numeric_round_cols] = export_df[numeric_round_cols].round(4)

export_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
feature_dictionary.to_csv(FEATURE_DICTIONARY_CSV, index=False, encoding="utf-8")
validation_report.to_csv(VALIDATION_CSV, index=False, encoding="utf-8")

with open(OUTPUT_CSV, "r", encoding="utf-8") as file:
    exported_rows = sum(1 for _ in file) - 1

assert exported_rows == len(export_df), "CSV export is incomplete or truncated."

print(f"Analytics-ready CSV : {OUTPUT_CSV}")
print(f"Rows exported       : {exported_rows:,}")
print(f"Columns exported    : {export_df.shape[1]}")
print(f"File size           : {OUTPUT_CSV.stat().st_size / 1024**2:,.2f} MB")
print(f"Feature dictionary  : {FEATURE_DICTIONARY_CSV}")
print(f"Validation report   : {VALIDATION_CSV}")

Analytics-ready CSV : /content/nexora_outputs/dataco_supply_chain_analytics_ready.csv
Rows exported       : 180,519
Columns exported    : 85
File size           : 105.01 MB
Feature dictionary  : /content/nexora_outputs/feature_dictionary.csv
Validation report   : /content/nexora_outputs/feature_engineering_validation.csv


## 13. Completion checklist

- [ ] Input contains approximately 180,000 rows.
- [ ] Output row count equals input row count.
- [ ] `order_item_id` is present, non-null, and unique.
- [ ] Delivery features are logically consistent.
- [ ] No infinite profit-margin values exist.
- [ ] Feature dictionary is exported.
- [ ] Validation report contains no unresolved failure.
- [ ] Analytics-ready CSV passes row-count read-back validation.